In [2]:
# 配置环境变量
from dotenv import load_dotenv
from openai import api_key

load_dotenv()
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.chat_models import init_chat_model
import os
import sys

#添加tools目录到路径
tools_dir = r'D:\code\LangChain-lesson\My-test\notebook\基础\tools'
sys.path.insert(0, tools_dir)
from weather import get_weather
from calculator import calculator

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")
if not api_key or api_key == "your_api_key_here":
    raise ValueError("请设置 DASHSCOPE_API_KEY 环境变量")
# 创建模型
model = init_chat_model(
    model="qwen-max",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)


# 执行循环最佳实践

In [24]:
def example_best_practices():
    print("\n" + "="*70)
    print("执行循环最佳实践")
    print("="*70)

    system_prompt="你是一个有帮助的助手"
    agent = create_agent(
        model=model,
        tools=[calculator],
        system_prompt=system_prompt,
    )
    try:
        response = agent.invoke({
            "messages": [
                {"role": "user", "content": "计算5 加 3"}
            ]
        })
        # 获取最终答案
        final_answer = response['messages'][-1].content
        print(f"最终答案: {final_answer}")

        # 查看使用的工具
        used_tools = [
            msg.tool_calls[0]['name']
            for msg in response['messages']
            if hasattr(msg, 'tool_calls') and msg.tool_calls
        ]
        print(f"使用的工具: {used_tools}")

        # 流式输出
        print("\n[流式输出演示]")
        input_text = "计算 10 乘以 5"
        step = 0
        stream = agent.stream({
            "messages": [
                {"role": "user", "content": input_text}
            ]
        })
        for chunk in stream:
            step += 1
            print(f"步骤 {step}: ")
            if 'model' in chunk and chunk['model'].get('messages'):
                latest = chunk['model']['messages'][-1]
                msg_type = latest.__class__.__name__
                print(f"类型：{msg_type}")
                if hasattr(latest, 'content') and latest.content:
                    content_preview = latest.content[:50] if len(latest.content) > 50 else latest.content
                    print(f"内容：{content_preview}...")
                elif hasattr(latest, 'tool_calls') and latest.tool_calls:
                    print(f"工具调用：{latest.tool_calls[0]['name']}")
            elif 'tools' in chunk and chunk['tools'].get('messages'):
                latest = chunk['tools']['messages'][-1]
                msg_type = latest.__class__.__name__
                print(f"  类型：{msg_type}")
                print(f"  工具结果：{latest.content}")
        # 调试时查看完整历史
        print("\n[完整历史]")
        response = agent.invoke({
            "messages": [
                {"role": "user", "content": "先计算 8 加 2，然后告诉我结果"}
            ]
        })
        for i,msg in enumerate(response['messages']):
            print(f"消息 {i}: {msg.__class__.__name__}")
            if hasattr(msg, 'content') and msg.content:
                print(f"内容：{msg.content}")
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                print(f"工具调用：{msg.tool_calls}")
            if hasattr(msg, 'tool_call_id'):
                print(f"工具调用 ID: {msg.tool_call_id}")
                print(f"工具结果：{msg.content}")
    except Exception as e:
        print(f"错误: {e}")

In [25]:
def main():
    print(f"\n" + "="*70)
    print(" LangChain 1.0 - agent_loop")
    print("="*70)

    try:
        example_best_practices()

        print("\n" + "="*70)
        print(" 完成！")
        print("="*70)

    except KeyboardInterrupt:
        print("\n程序中断")
    except Exception as e:
        print(f"错误: {e}")
        import traceback
        traceback.print_exc()

In [26]:
if __name__ == "__main__":
    main()


 LangChain 1.0 - agent_loop

执行循环最佳实践
最终答案: 5 加上 3 的结果是 8.0。
使用的工具: ['calculator']

[流式输出演示]
步骤 1: 
类型：AIMessage
工具调用：calculator
步骤 2: 
  类型：ToolMessage
  工具结果：10.0 multiply 5.0 = 50.0
步骤 3: 
类型：AIMessage
内容：计算 10 乘以 5 的结果是 50.0。...

[完整历史]
消息 0: HumanMessage
内容：先计算 8 加 2，然后告诉我结果
消息 1: AIMessage
工具调用：[{'name': 'calculator', 'args': {'operation': 'add', 'a': 8, 'b': 2}, 'id': 'call_de9ac55b9d4443d796313d', 'type': 'tool_call'}]
消息 2: ToolMessage
内容：8.0 add 2.0 = 10.0
工具调用 ID: call_de9ac55b9d4443d796313d
工具结果：8.0 add 2.0 = 10.0
消息 3: AIMessage
内容：计算 8 加 2 的结果是 10.0。

 完成！
